# Goal 4B — Human–machine item mapping atlas
## Exploratory cross-modal mapping and rater-stratified diagnostics

This notebook extends the finalized family-level correspondence analysis without changing its inferential conclusions. It asks how each retained signal indicator maps onto each individual human annotation item.

The hierarchy is explicit:

- the finalized family-level models remain the manuscript analysis;
- the adjusted item-level atlas uses logistic models with participant-clustered sandwich covariance, adjusted for rater, cohort, and recording duration;
- rater-stratified models are descriptive diagnostics because each production recording was rated by only one research assistant;
- sparse human items and sparse machine states are retained as unsupported or descriptive rather than forced into unstable models;
- task-adherence items are analyzed separately as negative-control outcomes;
- no human label is treated as physical ground truth;
- no imputation, family score, or data-driven feature selection is introduced.


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import platform
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Sequence

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.stats.multitest import multipletests
from IPython.display import display

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

pd.set_option("display.max_columns", 260)
pd.set_option("display.max_rows", 260)
pd.set_option("display.width", 280)

FINAL_RUN = True
SEED = 20260805

EXPECTED_FROZEN_RECORDINGS = 519
EXPECTED_FROZEN_PARTICIPANTS = 224
EXPECTED_HUMAN_RECORDINGS = 516
EXPECTED_HUMAN_ITEMS = 28
EXPECTED_MACHINE_FEATURES = 19
EXPECTED_RATERS = ["Abbas", "Liya", "Samaana", "Samara"]

# Joint item-atlas support gates.
MIN_JOINT_RECORDINGS = 100
MIN_JOINT_POSITIVES = 20
MIN_JOINT_NEGATIVES = 20
MIN_JOINT_POSITIVE_PARTICIPANTS = 10
MIN_JOINT_NEGATIVE_PARTICIPANTS = 10
MIN_JOINT_POSITIVE_RATERS = 2
MIN_BINARY_MACHINE_POSITIVES = 10
MIN_BINARY_MACHINE_ZEROS = 20

# Rater-stratified diagnostic gates.
MIN_RATER_RECORDINGS = 50
MIN_RATER_POSITIVES = 10
MIN_RATER_NEGATIVES = 20
MIN_RATER_POSITIVE_PARTICIPANTS = 8
MIN_RATER_NEGATIVE_PARTICIPANTS = 12
MIN_RATER_BINARY_POSITIVES = 8
MIN_RATER_BINARY_ZEROS = 20

MAX_ABSOLUTE_STABLE_PARAMETER = 25.0
REGRESSION_TEST_ATOL = 1e-7
REGRESSION_TEST_RTOL = 1e-6

def locate_analysis_root() -> Path:
    configured = os.environ.get("PAPER1_ANALYSIS_ROOT", "").strip()
    if configured:
        return Path(configured).expanduser().resolve()
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the analysis repository root")

ANALYSIS_ROOT = locate_analysis_root()
DEFAULT_PAPER_ROOT = ANALYSIS_ROOT.parent
PAPER_ROOT = Path(os.environ.get("PAPER1_ROOT", DEFAULT_PAPER_ROOT))
PIPELINE_ROOT = Path(os.environ.get("PAPER1_PIPELINE_ROOT", PAPER_ROOT / "Paper_1_Final_Cleanup"))
FAMILY_FREEZE_DIR = Path(
    os.environ.get(
        "PAPER1_FAMILY_FREEZE_DIR",
        PIPELINE_ROOT / "outputs" / "features",
    )
)

STAGE1_ROOT = (
    ANALYSIS_ROOT
    / "outputs"
    / "goal3_human_correspondence"
    / "stage1_annotation_audit"
)
STAGE2_ROOT = (
    ANALYSIS_ROOT
    / "outputs"
    / "goal3_human_correspondence"
    / "stage2_construct_matched_correspondence"
)
OUTPUT_ROOT = (
    ANALYSIS_ROOT
    / "outputs"
    / "goal3_human_correspondence"
    / "stage3_item_mapping_atlas"
)
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
FIGURE_SOURCE_DIR = OUTPUT_ROOT / "figures" / "source_data"
LOG_DIR = OUTPUT_ROOT / "logs"
MANIFEST_DIR = OUTPUT_ROOT / "manifest"

for directory in [TABLE_DIR, FIGURE_DIR, FIGURE_SOURCE_DIR, LOG_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FAMILY_ORDER = ["QADD", "QGAIN", "QREV", "QCHAN", "QDIST", "QTEMP", "TASK"]
MACHINE_FAMILY_ORDER = ["QADD", "QGAIN", "QREV", "QCHAN", "QDIST", "QTEMP"]

print("PAPER_ROOT:", PAPER_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("FINAL_RUN:", FINAL_RUN)


## 1. Shared utilities and numerical safeguards

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.casefold()
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix in {".parquet", ".pq"}:
        try:
            return pd.read_parquet(path)
        except ImportError:
            fallback = path.with_suffix(".csv")
            if fallback.exists():
                return pd.read_csv(fallback, low_memory=False)
            raise
    raise ValueError(f"Unsupported table type: {path}")


def resolve_table(
    root: Path,
    exact_relatives: Sequence[str],
    patterns: Sequence[str],
    *,
    label: str,
) -> Path:
    for relative in exact_relatives:
        candidate = root / relative
        if candidate.exists():
            return candidate

    matches: list[Path] = []
    for pattern in patterns:
        matches.extend(path for path in root.rglob(pattern) if path.is_file())

    matches = sorted(
        {path.resolve() for path in matches},
        key=lambda path: (
            0 if path.suffix.casefold() in {".parquet", ".pq"} else 1,
            "checkpoint" in str(path).casefold(),
            "source_candidate" in str(path).casefold(),
            len(str(path)),
            str(path).casefold(),
        ),
    )
    matches = [path for path in matches if "checkpoint" not in str(path).casefold()] or matches
    if not matches:
        raise FileNotFoundError(f"Could not locate {label} below {root}.")
    return matches[0]


def canonical_recording_id(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    text = str(value).strip().replace("\\", "/")
    name = text.rsplit("/", 1)[-1]
    lower_name = name.casefold()
    for suffix in [
        "_segments.csv", ".segments.csv", ".csv", ".wav", ".mp3",
        ".m4a", ".mp4", ".flac",
    ]:
        if lower_name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return re.sub(r"\s+", "", name).casefold()


def canonical_participant_id(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    text = str(value).strip()
    if text.casefold() in {"", "nan", "none", "null", "na"}:
        return ""
    return re.sub(r"\s+", "", text).casefold()


def boolean_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return (
        series.astype("string").str.strip().str.casefold()
        .map({"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False})
        .fillna(False).astype(bool)
    )


def transform_feature(series: pd.Series, transform: str) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    if transform == "none":
        return values
    if transform == "log1p":
        if (values.dropna() < 0).any():
            raise ValueError("log1p requested for a feature containing negative values.")
        return np.log1p(values)
    if transform == "asinh":
        return np.arcsinh(values)
    raise ValueError(f"Unknown transform: {transform}")


def robust_scale(series: pd.Series) -> tuple[pd.Series, float, float]:
    values = pd.to_numeric(series, errors="coerce")
    median = float(values.median(skipna=True))
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = float(q3 - q1)
    if not np.isfinite(iqr) or iqr <= 0:
        return pd.Series(np.nan, index=series.index, dtype=float), median, iqr
    return (values - median) / iqr, median, iqr


def export_dataframe(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def bh_values(values: pd.Series) -> pd.Series:
    output = pd.Series(np.nan, index=values.index, dtype=float)
    mask = values.notna() & np.isfinite(values)
    if mask.any():
        output.loc[mask] = multipletests(values.loc[mask].astype(float), method="fdr_bh")[1]
    return output


def apply_grouped_bh(
    frame: pd.DataFrame,
    group_columns: list[str],
    p_column: str,
    output_column: str,
) -> pd.DataFrame:
    result = frame.copy()
    result[output_column] = np.nan
    for _, group in result.groupby(group_columns, dropna=False, sort=False):
        result.loc[group.index, output_column] = bh_values(group[p_column])
    return result


def stable_gee_fit(
    data: pd.DataFrame,
    formula: str,
    *,
    group_column: str = "participant_id",
) -> tuple[Any | None, dict[str, Any]]:
    audit = {
        "model_status": "failed",
        "model_error": "",
        "model_converged": False,
        "numerical_warnings": "",
        "maximum_absolute_parameter": np.nan,
    }
    try:
        model = smf.gee(
            formula=formula,
            groups=group_column,
            data=data,
            family=sm.families.Binomial(),
            cov_struct=Exchangeable(),
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", RuntimeWarning)
            result = model.fit(maxiter=250)

        warning_messages = sorted({
            str(warning.message)
            for warning in caught
            if issubclass(warning.category, RuntimeWarning)
        })
        audit["numerical_warnings"] = " | ".join(warning_messages)
        audit["model_converged"] = bool(getattr(result, "converged", True))

        params = pd.to_numeric(result.params, errors="coerce")
        errors = pd.to_numeric(result.bse, errors="coerce")
        finite = bool(np.isfinite(params).all() and np.isfinite(errors).all())
        max_abs = float(np.max(np.abs(params))) if finite and len(params) else np.nan
        audit["maximum_absolute_parameter"] = max_abs
        bad_warning = any(
            "overflow" in message.casefold() or "invalid value" in message.casefold()
            for message in warning_messages
        )
        stable = bool(
            audit["model_converged"]
            and finite
            and not bad_warning
            and np.isfinite(max_abs)
            and max_abs <= MAX_ABSOLUTE_STABLE_PARAMETER
        )
        if not stable:
            audit["model_status"] = "unstable_numerical_fit"
            audit["model_error"] = (
                "Non-convergence, non-finite estimates, numerical overflow, "
                "or an implausibly large coefficient."
            )
            return None, audit

        audit["model_status"] = "fit"
        return result, audit

    except Exception as exc:
        audit["model_error"] = f"{type(exc).__name__}: {exc}"
        return None, audit


def extract_effect(result: Any, term: str = "machine_predictor") -> dict[str, float]:
    estimate = float(result.params[term])
    standard_error = float(result.bse[term])
    p_value = float(result.pvalues[term])
    return {
        "estimate": estimate,
        "standard_error": standard_error,
        "wald_z": estimate / standard_error if standard_error > 0 else np.nan,
        "ci95_low": estimate - 1.96 * standard_error,
        "ci95_high": estimate + 1.96 * standard_error,
        "odds_ratio": float(np.exp(estimate)),
        "odds_ratio_ci95_low": float(np.exp(estimate - 1.96 * standard_error)),
        "odds_ratio_ci95_high": float(np.exp(estimate + 1.96 * standard_error)),
        "p_value": p_value,
    }



def stable_clustered_logistic_fit(
    data: pd.DataFrame,
    formula: str,
    *,
    cluster_column: str = "participant_id",
) -> tuple[Any | None, dict[str, Any]]:
    """Logistic GLM with participant-clustered sandwich covariance."""
    audit = {
        "model_status": "failed",
        "model_error": "",
        "model_converged": False,
        "numerical_warnings": "",
        "maximum_absolute_parameter": np.nan,
    }
    try:
        model = smf.glm(
            formula=formula,
            data=data,
            family=sm.families.Binomial(),
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", RuntimeWarning)
            result = model.fit(
                maxiter=250,
                disp=0,
                cov_type="cluster",
                cov_kwds={"groups": data[cluster_column]},
            )

        warning_messages = sorted({
            str(warning.message)
            for warning in caught
            if issubclass(warning.category, RuntimeWarning)
        })
        audit["numerical_warnings"] = " | ".join(warning_messages)
        audit["model_converged"] = bool(getattr(result, "converged", True))
        params = pd.to_numeric(result.params, errors="coerce")
        errors = pd.to_numeric(result.bse, errors="coerce")
        finite = bool(np.isfinite(params).all() and np.isfinite(errors).all())
        max_abs = float(np.max(np.abs(params))) if finite and len(params) else np.nan
        audit["maximum_absolute_parameter"] = max_abs
        bad_warning = any(
            "overflow" in message.casefold() or "invalid value" in message.casefold()
            for message in warning_messages
        )
        stable = bool(
            audit["model_converged"]
            and finite
            and not bad_warning
            and np.isfinite(max_abs)
            and max_abs <= MAX_ABSOLUTE_STABLE_PARAMETER
        )
        if not stable:
            audit["model_status"] = "unstable_numerical_fit"
            audit["model_error"] = (
                "Non-convergence, non-finite estimates, numerical overflow, "
                "or an implausibly large coefficient."
            )
            return None, audit
        audit["model_status"] = "fit"
        return result, audit
    except Exception as exc:
        audit["model_error"] = f"{type(exc).__name__}: {exc}"
        return None, audit


## 2. Load governed human and finalized Stage-2 inputs

In [ ]:
stage1_paths = {
    "item": STAGE1_ROOT / "tables" / "humanqc_recording_item_analysis_population.csv",
    "family": STAGE1_ROOT / "tables" / "humanqc_recording_signal_family_analysis_population.csv",
    "population_metadata": STAGE1_ROOT / "tables" / "frozen_519_population_with_metadata.csv",
}
stage2_paths = {
    "feature_registry": STAGE2_ROOT / "tables" / "feature_registry.csv",
    "construct_map": STAGE2_ROOT / "tables" / "frozen_construct_map.csv",
    "primary_models": STAGE2_ROOT / "tables" / "primary_construct_matched_models.csv",
    "completion": STAGE2_ROOT / "logs" / "stage2_blocking_completion_checklist.csv",
    "manifest": STAGE2_ROOT / "manifest" / "stage2_run_manifest.json",
}

required_paths = {**stage1_paths, **stage2_paths}
missing = [str(path) for path in required_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Required governed inputs are missing:\n" + "\n".join(missing))

human_item = read_table(stage1_paths["item"])
human_family = read_table(stage1_paths["family"])
metadata_raw = read_table(stage1_paths["population_metadata"])
feature_registry = read_table(stage2_paths["feature_registry"])
construct_map = read_table(stage2_paths["construct_map"])
primary_reference = read_table(stage2_paths["primary_models"])
stage2_completion = read_table(stage2_paths["completion"])
stage2_manifest = json.loads(stage2_paths["manifest"].read_text(encoding="utf-8"))

if not stage2_completion["passed"].astype(bool).all():
    raise RuntimeError("The finalized Stage-2 analysis did not pass all blocking checks.")

if len(feature_registry) != EXPECTED_MACHINE_FEATURES:
    raise RuntimeError(
        f"Expected {EXPECTED_MACHINE_FEATURES} machine indicators; found {len(feature_registry)}."
    )

print("Human item rows:", len(human_item))
print("Machine indicators:", len(feature_registry))
print("Stage-2 completion checks:", len(stage2_completion))


## 3. Discover frozen family tables and reconstruct the 519-recording signal matrix

In [ ]:
family_paths = {
    "QADD": resolve_table(
        FAMILY_FREEZE_DIR,
        [
            r"families\02_QADD\features.parquet",
            r"additive_interference\qadd-v4.2.0\tables\qadd_v420_recording_features.parquet",
            r"additive_interference\qadd-v4.2.0\tables\qadd_v420_recording_features.csv",
        ],
        ["qadd_v420_recording_features.parquet", "qadd_v420_recording_features.csv"],
        label="QADD recording features",
    ),
    "QGAIN": resolve_table(
        FAMILY_FREEZE_DIR,
        [
            r"families\01_QGAIN\features.parquet",
            r"gain_dynamics\qgain-v4.1.0\tables\qgain_v410_recording_features.parquet",
            r"gain_dynamics\qgain-v4.1.0\tables\qgain_v410_recording_features.csv",
        ],
        ["qgain_v410_recording_features.parquet", "qgain_v410_recording_features.csv"],
        label="QGAIN recording features",
    ),
    "QREV": resolve_table(
        FAMILY_FREEZE_DIR,
        [
            r"families\03_QREV\features.parquet",
            r"reverberation\qrev-v4.0.0\tables\qrev_v400_analysis_features.parquet",
            r"reverberation\qrev-v4.0.0\tables\qrev_v400_analysis_features.csv",
        ],
        ["qrev_v400_analysis_features.parquet", "qrev_v400_analysis_features.csv"],
        label="QREV analysis features",
    ),
    "QCHAN": resolve_table(
        FAMILY_FREEZE_DIR,
        [
            r"families\04_QCHAN\features.parquet",
            r"channel_device\qchan-v4.0.0\tables\qchan_v400_analysis_features.parquet",
            r"channel_device\qchan-v4.0.0\tables\qchan_v400_analysis_features.csv",
        ],
        ["qchan_v400_analysis_features.parquet", "qchan_v400_analysis_features.csv"],
        label="QCHAN analysis features",
    ),
    "QDIST": resolve_table(
        FAMILY_FREEZE_DIR,
        [
            "families/05_QDIST/features.parquet",
            "nonlinear_distortion/qdist-v4.1.0/finalization/tables/qdist_v410_analysis_features.parquet",
            "nonlinear_distortion/qdist-v4.1.0/finalization/tables/qdist_v410_analysis_features.csv",
            "nonlinear_distortion/qdist-v4.1.0/tables/qdist_v410_analysis_features.parquet",
            "nonlinear_distortion/qdist-v4.1.0/tables/qdist_v410_analysis_features.csv",
        ],
        ["qdist_v410_analysis_features.parquet", "qdist_v410_analysis_features.csv"],
        label="QDIST analysis features",
    ),
    "QTEMP": resolve_table(
        FAMILY_FREEZE_DIR,
        [r"temporal_discontinuity\qtemp-v1.0.0-analytical-final-no-retained\tables\qtemp_v100_exploratory_features.csv"],
        ["qtemp_v100_exploratory_features.csv"],
        label="QTEMP exploratory features",
    ),
}

input_inventory = pd.DataFrame([
    *[
        {"input": name, "path": str(path), "sha256": sha256_file(path), "bytes": path.stat().st_size}
        for name, path in required_paths.items()
    ],
    *[
        {"input": family, "path": str(path), "sha256": sha256_file(path), "bytes": path.stat().st_size}
        for family, path in family_paths.items()
    ],
])
export_dataframe(input_inventory, MANIFEST_DIR / "input_inventory.csv")
display(input_inventory)


def select_family_columns(frame: pd.DataFrame, family: str) -> pd.DataFrame:
    id_column = (
        "logical_recording_id" if "logical_recording_id" in frame.columns
        else "recording_id" if "recording_id" in frame.columns
        else None
    )
    if id_column is None:
        raise KeyError(f"{family}: no recording ID column was found.")
    selected_columns = [
        id_column,
        *[column for column in frame.columns if str(column).casefold().startswith(family.casefold())],
    ]
    selected = frame.loc[:, list(dict.fromkeys(selected_columns))].copy()
    selected["recording_id"] = selected[id_column].map(canonical_recording_id)
    if id_column != "recording_id":
        selected = selected.drop(columns=id_column)
    if selected["recording_id"].duplicated().any():
        raise RuntimeError(f"{family}: duplicate recording IDs in frozen table.")
    return selected

family_tables = {
    family: select_family_columns(read_table(path), family)
    for family, path in family_paths.items()
}
id_sets = {family: set(frame["recording_id"].dropna()) for family, frame in family_tables.items()}
intersection_ids = set.intersection(*id_sets.values())
union_ids = set.union(*id_sets.values())
if len(intersection_ids) != EXPECTED_FROZEN_RECORDINGS or len(union_ids) != EXPECTED_FROZEN_RECORDINGS:
    raise RuntimeError(
        "The six family tables do not define one common 519-recording population. "
        f"Intersection={len(intersection_ids)}, union={len(union_ids)}."
    )

signal_matrix = pd.DataFrame({"recording_id": sorted(intersection_ids)})
for family in MACHINE_FAMILY_ORDER:
    signal_matrix = signal_matrix.merge(
        family_tables[family], on="recording_id", how="left", validate="one_to_one"
    )

missing_features = [
    feature for feature in feature_registry["feature"]
    if feature not in signal_matrix.columns
]
if missing_features:
    raise KeyError(f"Missing frozen feature columns: {missing_features}")

metadata = metadata_raw.copy()
required_metadata = {"recording_id", "participant_id", "cohort"}
if not required_metadata.issubset(metadata.columns):
    raise KeyError(f"Missing metadata columns: {sorted(required_metadata - set(metadata.columns))}")
metadata["recording_id"] = metadata["recording_id"].map(canonical_recording_id)
metadata["participant_id"] = metadata["participant_id"].map(canonical_participant_id)
metadata["cohort"] = metadata["cohort"].astype("string").str.strip().str.upper()
metadata = metadata.loc[metadata["recording_id"].isin(intersection_ids)].drop_duplicates("recording_id")

analysis_519 = metadata[["recording_id", "participant_id", "cohort"]].merge(
    signal_matrix, on="recording_id", how="inner", validate="one_to_one"
)
if len(analysis_519) != EXPECTED_FROZEN_RECORDINGS:
    raise RuntimeError("Frozen signal population was not reconstructed.")
if analysis_519["participant_id"].nunique() != EXPECTED_FROZEN_PARTICIPANTS:
    raise RuntimeError("Frozen participant count was not reproduced.")
if set(analysis_519["cohort"].dropna()) != {"ALS", "CONTROL"}:
    raise RuntimeError("Unexpected cohort values.")

print("Frozen recordings:", len(analysis_519))
print("Frozen participants:", analysis_519["participant_id"].nunique())


## 4. Freeze artifact-oriented indicator direction and scaling

In [ ]:
ARTIFACT_DIRECTION = {
    "qadd_pause_ac_level_dbfs_median": 1,
    "qadd_pause_level_iqr_db": 1,
    "qadd_speech_pause_level_contrast_db": -1,
    "qadd_mains_hum_comb_score_db": 1,
    "qgain_within_segment_iqr_db": 1,
    "qgain_between_segment_mad_db": 1,
    "qgain_abs_drift_db_per_min": 1,
    "qrev_tail_excess_100ms_db": 1,
    "qrev_tail_persistence_median_sec": 1,
    "qrev_srmr_norm": -1,
    "qchan_ltas_distance_db": 1,
    "qchan_rolloff95_deficit_hz": 1,
    "qchan_highband_ratio_deficit": 1,
    "qchan_tilt_steepening_db_per_oct": 1,
    "qdist_hard_clipped_sample_fraction": 1,
    "qdist_hard_clip_event_rate_per_min": 1,
    "qtemp_dropout_duration_fraction": 1,
    "qtemp_dropout_event_rate_per_min": 1,
    "qtemp_frozen_audio_duration_fraction": 1,
}

if set(ARTIFACT_DIRECTION) != set(feature_registry["feature"]):
    raise RuntimeError("Artifact-direction registry does not match the frozen feature registry.")

feature_registry = feature_registry.copy()
feature_registry["artifact_direction"] = feature_registry["feature"].map(ARTIFACT_DIRECTION)

scaling_rows = []
for row in feature_registry.itertuples(index=False):
    transformed = transform_feature(analysis_519[row.feature], row.transform)
    scaled, median, iqr = robust_scale(transformed)
    analysis_519[f"{row.feature}__transformed"] = transformed
    analysis_519[f"{row.feature}__z"] = scaled
    analysis_519[f"{row.feature}__positive"] = (
        analysis_519[row.feature].notna()
        & pd.to_numeric(analysis_519[row.feature], errors="coerce").gt(0)
    )
    if row.predictor_type == "continuous":
        analysis_519[f"{row.feature}__artifact"] = row.artifact_direction * scaled
    else:
        analysis_519[f"{row.feature}__artifact"] = analysis_519[f"{row.feature}__positive"].astype(float)
    scaling_rows.append({
        "feature": row.feature,
        "family": row.family,
        "label": row.label,
        "predictor_type": row.predictor_type,
        "transform": row.transform,
        "artifact_direction": row.artifact_direction,
        "n_available": int(transformed.notna().sum()),
        "transformed_median": median,
        "transformed_iqr": iqr,
        "positive_state_n": int(analysis_519[f"{row.feature}__positive"].sum()),
    })

scaling_registry = pd.DataFrame(scaling_rows)
export_dataframe(scaling_registry, TABLE_DIR / "atlas_scaling_registry.csv")
export_dataframe(feature_registry, TABLE_DIR / "atlas_feature_registry.csv")
display(scaling_registry)


## 5. Build the complete human item inventory and production outcome matrix

In [ ]:
human_item = human_item.copy()
human_item["recording_id"] = human_item["recording_id"].map(canonical_recording_id)
human_item["participant_id"] = human_item["participant_id"].map(canonical_participant_id)
human_item["rater"] = human_item["rater"].astype("string").str.strip()
human_item["cohort"] = human_item["cohort"].astype("string").str.strip().str.upper()
human_item["present"] = boolean_series(human_item["present"])

item_metadata = (
    human_item[["signal_family", "source_family", "item"]]
    .drop_duplicates()
    .assign(
        family_order=lambda frame: frame["signal_family"].map({family: index for index, family in enumerate(FAMILY_ORDER)})
    )
    .sort_values(["family_order", "source_family", "item"])
    .drop(columns="family_order")
    .reset_index(drop=True)
)

if len(item_metadata) != EXPECTED_HUMAN_ITEMS:
    raise RuntimeError(f"Expected {EXPECTED_HUMAN_ITEMS} human items; found {len(item_metadata)}.")

recording_item_counts = human_item.groupby(["recording_id", "item"]).size()
if (recording_item_counts > 1).any():
    raise RuntimeError("A production recording contains duplicate rows for a human item.")

production_ids = set(human_item["recording_id"])
if len(production_ids) != EXPECTED_HUMAN_RECORDINGS:
    raise RuntimeError(
        f"Expected {EXPECTED_HUMAN_RECORDINGS} production-annotated recordings; found {len(production_ids)}."
    )

item_prevalence_rows = []
for item_row in item_metadata.itertuples(index=False):
    subset = human_item.loc[human_item["item"].eq(item_row.item)].copy()
    row = {
        "signal_family": item_row.signal_family,
        "source_family": item_row.source_family,
        "item": item_row.item,
        "recordings": subset["recording_id"].nunique(),
        "participants": subset["participant_id"].nunique(),
        "positives": int(subset["present"].sum()),
        "positive_participants": subset.loc[subset["present"], "participant_id"].nunique(),
        "positive_raters": subset.loc[subset["present"], "rater"].nunique(),
        "prevalence": float(subset["present"].mean()),
    }
    for rater in EXPECTED_RATERS:
        rater_subset = subset.loc[subset["rater"].eq(rater)]
        row[f"n_{rater}"] = len(rater_subset)
        row[f"positive_{rater}"] = int(rater_subset["present"].sum())
        row[f"prevalence_{rater}"] = float(rater_subset["present"].mean()) if len(rater_subset) else np.nan
    item_prevalence_rows.append(row)

item_prevalence = pd.DataFrame(item_prevalence_rows)
export_dataframe(item_prevalence, TABLE_DIR / "human_item_prevalence_and_support.csv")
display(item_prevalence)

LOG_DURATION_CENTER = float(np.log(human_item[["recording_id", "recording_duration_sec"]].drop_duplicates()["recording_duration_sec"]).median())
print("Centered log-duration reference:", LOG_DURATION_CENTER)


## 6. Model preparation and support gates

In [ ]:
def extract_item_outcome(item: str, allowed_rater: str | None = None) -> pd.DataFrame:
    subset = human_item.loc[
        human_item["item"].eq(item),
        ["recording_id", "participant_id", "cohort", "rater", "recording_duration_sec", "present"],
    ].copy()
    if allowed_rater is not None:
        subset = subset.loc[subset["rater"].eq(allowed_rater)].copy()
    subset = subset.rename(columns={"present": "human_present"})
    subset["human_present"] = subset["human_present"].astype(int)
    return subset


def extract_family_outcome(family: str) -> pd.DataFrame:
    subset = human_family.loc[
        human_family["signal_family"].eq(family),
        ["recording_id", "participant_id", "cohort", "rater", "recording_duration_sec", "present"],
    ].copy()
    subset["recording_id"] = subset["recording_id"].map(canonical_recording_id)
    subset["participant_id"] = subset["participant_id"].map(canonical_participant_id)
    subset["cohort"] = subset["cohort"].astype("string").str.strip().str.upper()
    subset["rater"] = subset["rater"].astype("string").str.strip()
    subset["human_present"] = boolean_series(subset["present"]).astype(int)
    return subset.drop(columns="present")


def prepare_mapping_data(
    human_outcome: pd.DataFrame,
    feature_row: Any,
    *,
    artifact_oriented: bool,
) -> pd.DataFrame:
    data = analysis_519.merge(
        human_outcome,
        on=["recording_id", "participant_id", "cohort"],
        how="inner",
        validate="one_to_one",
    )
    if artifact_oriented:
        data["machine_predictor"] = data[f"{feature_row.feature}__artifact"]
    elif feature_row.predictor_type == "continuous":
        data["machine_predictor"] = data[f"{feature_row.feature}__z"]
    else:
        data["machine_predictor"] = data[f"{feature_row.feature}__positive"].astype(float)

    data["log_recording_duration_centered"] = (
        np.log(data["recording_duration_sec"]) - LOG_DURATION_CENTER
    )
    data = data[[
        "recording_id", "participant_id", "cohort", "rater",
        "recording_duration_sec", "log_recording_duration_centered",
        "human_present", "machine_predictor",
    ]].dropna()
    data["human_present"] = data["human_present"].astype(int)
    data["rater"] = data["rater"].astype("category")
    data["cohort"] = pd.Categorical(data["cohort"], categories=["CONTROL", "ALS"])
    return data


def support_gate(data: pd.DataFrame, predictor_type: str, *, rater_stratified: bool) -> tuple[bool, dict[str, Any]]:
    positives = data.loc[data["human_present"].eq(1)]
    negatives = data.loc[data["human_present"].eq(0)]
    predictor_positive = int(data["machine_predictor"].gt(0).sum())
    predictor_zero = int(data["machine_predictor"].eq(0).sum())
    reasons = []

    if rater_stratified:
        thresholds = {
            "recordings": MIN_RATER_RECORDINGS,
            "positives": MIN_RATER_POSITIVES,
            "negatives": MIN_RATER_NEGATIVES,
            "positive_participants": MIN_RATER_POSITIVE_PARTICIPANTS,
            "negative_participants": MIN_RATER_NEGATIVE_PARTICIPANTS,
        }
    else:
        thresholds = {
            "recordings": MIN_JOINT_RECORDINGS,
            "positives": MIN_JOINT_POSITIVES,
            "negatives": MIN_JOINT_NEGATIVES,
            "positive_participants": MIN_JOINT_POSITIVE_PARTICIPANTS,
            "negative_participants": MIN_JOINT_NEGATIVE_PARTICIPANTS,
        }

    metrics = {
        "recordings": len(data),
        "participants": data["participant_id"].nunique(),
        "human_positives": len(positives),
        "human_negatives": len(negatives),
        "positive_participants": positives["participant_id"].nunique(),
        "negative_participants": negatives["participant_id"].nunique(),
        "positive_raters": positives["rater"].nunique(),
        "predictor_unique": data["machine_predictor"].nunique(),
        "predictor_positive": predictor_positive,
        "predictor_zero": predictor_zero,
    }

    for key, threshold in thresholds.items():
        observed_key = {
            "positives": "human_positives",
            "negatives": "human_negatives",
        }.get(key, key)
        if metrics[observed_key] < threshold:
            reasons.append(f"{key}<{threshold}")

    if metrics["predictor_unique"] < 2:
        reasons.append("degenerate predictor")

    if not rater_stratified and metrics["positive_raters"] < MIN_JOINT_POSITIVE_RATERS:
        reasons.append(f"positive_raters<{MIN_JOINT_POSITIVE_RATERS}")

    if predictor_type != "continuous":
        minimum_positive = MIN_RATER_BINARY_POSITIVES if rater_stratified else MIN_BINARY_MACHINE_POSITIVES
        minimum_zero = MIN_RATER_BINARY_ZEROS if rater_stratified else MIN_BINARY_MACHINE_ZEROS
        if predictor_positive < minimum_positive:
            reasons.append(f"machine_positive<{minimum_positive}")
        if predictor_zero < minimum_zero:
            reasons.append(f"machine_zero<{minimum_zero}")

    return len(reasons) == 0, {**metrics, "support_reason": " | ".join(reasons)}


## 7. Complete descriptive correlation atlas, jointly and by rater

In [ ]:
# All 28 human items × all 19 machine indicators are retained here.
# These Spearman correlations are descriptive: they do not adjust for
# repeated participants, rater allocation, cohort, or duration.
raw_rows = []

for item_row in item_metadata.itertuples(index=False):
    human_all = extract_item_outcome(item_row.item)

    for feature_row in feature_registry.itertuples(index=False):
        data_all = prepare_mapping_data(
            human_all,
            feature_row,
            artifact_oriented=True,
        )

        def correlation_record(frame: pd.DataFrame) -> tuple[float, float]:
            if (
                len(frame) < 3
                or frame["human_present"].nunique() < 2
                or frame["machine_predictor"].nunique() < 2
            ):
                return np.nan, np.nan
            rho, p_value = stats.spearmanr(
                frame["machine_predictor"],
                frame["human_present"],
                nan_policy="omit",
            )
            return float(rho), float(p_value)

        rho, p_value = correlation_record(data_all)
        raw_rows.append({
            "scope": "joint_unadjusted",
            "rater": "Joint",
            "human_signal_family": item_row.signal_family,
            "human_source_family": item_row.source_family,
            "human_item": item_row.item,
            "machine_family": feature_row.family,
            "feature": feature_row.feature,
            "feature_label": feature_row.label,
            "predictor_type": feature_row.predictor_type,
            "family_match": bool(item_row.signal_family == feature_row.family),
            "task_negative_control": bool(item_row.signal_family == "TASK"),
            "recordings": len(data_all),
            "participants": data_all["participant_id"].nunique(),
            "human_positives": int(data_all["human_present"].sum()),
            "human_negatives": int((1 - data_all["human_present"]).sum()),
            "predictor_unique": data_all["machine_predictor"].nunique(),
            "spearman_rho": rho,
            "p_value": p_value,
        })

        for rater in EXPECTED_RATERS:
            data_rater = data_all.loc[
                data_all["rater"].astype(str).eq(rater)
            ].copy()
            rater_rho, rater_p = correlation_record(data_rater)
            raw_rows.append({
                "scope": "rater_unadjusted",
                "rater": rater,
                "human_signal_family": item_row.signal_family,
                "human_source_family": item_row.source_family,
                "human_item": item_row.item,
                "machine_family": feature_row.family,
                "feature": feature_row.feature,
                "feature_label": feature_row.label,
                "predictor_type": feature_row.predictor_type,
                "family_match": bool(item_row.signal_family == feature_row.family),
                "task_negative_control": bool(item_row.signal_family == "TASK"),
                "recordings": len(data_rater),
                "participants": data_rater["participant_id"].nunique(),
                "human_positives": int(data_rater["human_present"].sum()),
                "human_negatives": int((1 - data_rater["human_present"]).sum()),
                "predictor_unique": data_rater["machine_predictor"].nunique(),
                "spearman_rho": rater_rho,
                "p_value": rater_p,
            })

raw_correlation_atlas = pd.DataFrame(raw_rows)
expected_raw_rows = (
    EXPECTED_HUMAN_ITEMS
    * EXPECTED_MACHINE_FEATURES
    * (1 + len(EXPECTED_RATERS))
)
if len(raw_correlation_atlas) != expected_raw_rows:
    raise RuntimeError(
        "The descriptive correlation atlas is not the complete "
        "item-by-feature-by-scope Cartesian product."
    )

raw_correlation_atlas = apply_grouped_bh(
    raw_correlation_atlas,
    ["scope", "rater"],
    "p_value",
    "q_within_scope",
)

export_dataframe(
    raw_correlation_atlas,
    TABLE_DIR / "complete_raw_item_feature_correlations_joint_and_by_rater.csv",
)

display(
    raw_correlation_atlas.loc[
        raw_correlation_atlas["scope"].eq("joint_unadjusted")
        & raw_correlation_atlas["spearman_rho"].notna()
    ]
    .assign(abs_rho=lambda frame: frame["spearman_rho"].abs())
    .sort_values("abs_rho", ascending=False)
    .head(40)
)


## 8. Adjusted within-family item mapping with participant-clustered logistic models

In [ ]:
# Inferential modeling is restricted to acquisition items and indicators
# from the same quality family. This avoids turning the all-pair atlas into
# an indiscriminate multiple-testing exercise.
adjusted_rows = []

for item_row in item_metadata.loc[
    item_metadata["signal_family"].isin(MACHINE_FAMILY_ORDER)
].itertuples(index=False):
    human = extract_item_outcome(item_row.item)
    matched_features = feature_registry.loc[
        feature_registry["family"].eq(item_row.signal_family)
    ]

    for feature_row in matched_features.itertuples(index=False):
        data = prepare_mapping_data(
            human,
            feature_row,
            artifact_oriented=True,
        )
        support_pass, support = support_gate(
            data,
            feature_row.predictor_type,
            rater_stratified=False,
        )
        base = {
            "human_signal_family": item_row.signal_family,
            "human_source_family": item_row.source_family,
            "human_item": item_row.item,
            "machine_family": feature_row.family,
            "feature": feature_row.feature,
            "feature_label": feature_row.label,
            "predictor_type": feature_row.predictor_type,
            "support_pass": support_pass,
            **support,
        }

        if not support_pass:
            adjusted_rows.append({
                **base,
                "model_status": "support_gate_failed",
                "model_error": support["support_reason"],
            })
            continue

        result, audit = stable_clustered_logistic_fit(
            data,
            "human_present ~ machine_predictor + C(rater) + C(cohort) "
            "+ log_recording_duration_centered",
        )
        if result is None:
            adjusted_rows.append({**base, **audit})
            continue

        adjusted_rows.append({
            **base,
            **audit,
            **extract_effect(result),
        })

adjusted_within_family = pd.DataFrame(adjusted_rows)
adjusted_within_family["q_global_within_family_atlas"] = bh_values(
    adjusted_within_family["p_value"]
)
adjusted_within_family = apply_grouped_bh(
    adjusted_within_family,
    ["human_signal_family"],
    "p_value",
    "q_within_quality_family",
)

export_dataframe(
    adjusted_within_family,
    TABLE_DIR / "adjusted_within_family_item_feature_models.csv",
)

display(
    adjusted_within_family.loc[
        adjusted_within_family["model_status"].eq("fit")
    ]
    .sort_values("q_global_within_family_atlas")
)


## 9. Independent regression test against the finalized Stage-2 models

In [ ]:
verification_rows = []
feature_lookup = feature_registry.set_index("feature")

for reference_row in primary_reference.loc[
    primary_reference["model_status"].eq("fit")
].itertuples(index=False):
    map_row = construct_map.loc[
        construct_map["pair_id"].eq(reference_row.pair_id)
    ].iloc[0]
    feature_row = next(
        feature_registry.loc[
            feature_registry["feature"].eq(reference_row.feature)
        ].itertuples(index=False)
    )

    if map_row["human_level"] == "signal_family":
        human = extract_family_outcome(map_row["human_value"])
    elif map_row["human_level"] == "item":
        human = extract_item_outcome(map_row["human_value"])
    else:
        raise ValueError(map_row["human_level"])

    data = prepare_mapping_data(
        human,
        feature_row,
        artifact_oriented=False,
    )
    result, audit = stable_gee_fit(
        data,
        "human_present ~ machine_predictor + C(rater) + C(cohort) "
        "+ log_recording_duration_centered",
    )
    if result is None:
        verification_rows.append({
            "pair_id": reference_row.pair_id,
            "passed": False,
            "detail": audit["model_error"],
        })
        continue

    effect = extract_effect(result)
    estimate_difference = abs(
        effect["estimate"] - float(reference_row.estimate)
    )
    se_difference = abs(
        effect["standard_error"] - float(reference_row.standard_error)
    )
    n_match = len(data) == int(reference_row.recordings)
    positives_match = (
        int(data["human_present"].sum())
        == int(reference_row.human_positives)
    )
    passed = bool(
        np.isclose(
            effect["estimate"],
            float(reference_row.estimate),
            atol=REGRESSION_TEST_ATOL,
            rtol=REGRESSION_TEST_RTOL,
        )
        and np.isclose(
            effect["standard_error"],
            float(reference_row.standard_error),
            atol=REGRESSION_TEST_ATOL,
            rtol=REGRESSION_TEST_RTOL,
        )
        and n_match
        and positives_match
    )
    verification_rows.append({
        "pair_id": reference_row.pair_id,
        "reference_estimate": float(reference_row.estimate),
        "recomputed_estimate": effect["estimate"],
        "absolute_estimate_difference": estimate_difference,
        "reference_standard_error": float(reference_row.standard_error),
        "recomputed_standard_error": effect["standard_error"],
        "absolute_standard_error_difference": se_difference,
        "recordings_match": n_match,
        "positives_match": positives_match,
        "passed": passed,
        "detail": "",
    })

regression_verification = pd.DataFrame(verification_rows)
export_dataframe(
    regression_verification,
    LOG_DIR / "stage2_regression_verification.csv",
)
display(regression_verification)

if not regression_verification["passed"].all():
    raise RuntimeError(
        "The atlas data/model pipeline did not reproduce finalized "
        "Stage-2 estimates."
    )


## 10. Rater-stratified descriptive consistency

In [ ]:
# The production raters evaluated largely non-overlapping recordings.
# Therefore, the per-rater layer remains a descriptive correlation audit.
# We summarize whether estimable rater-specific correlations have the same
# sign as the joint descriptive correlation, without treating differences
# as tests of rater performance.

rater_consistency_rows = []

joint_raw = raw_correlation_atlas.loc[
    raw_correlation_atlas["scope"].eq("joint_unadjusted")
].copy()

for joint_row in joint_raw.itertuples(index=False):
    rater_subset = raw_correlation_atlas.loc[
        raw_correlation_atlas["scope"].eq("rater_unadjusted")
        & raw_correlation_atlas["human_item"].eq(joint_row.human_item)
        & raw_correlation_atlas["feature"].eq(joint_row.feature)
        & raw_correlation_atlas["spearman_rho"].notna()
    ].copy()

    joint_sign = (
        int(np.sign(joint_row.spearman_rho))
        if np.isfinite(joint_row.spearman_rho)
        else 0
    )

    same_direction = (
        np.sign(rater_subset["spearman_rho"]) == joint_sign
        if joint_sign != 0
        else pd.Series(False, index=rater_subset.index)
    )

    rater_consistency_rows.append({
        "human_signal_family": joint_row.human_signal_family,
        "human_item": joint_row.human_item,
        "machine_family": joint_row.machine_family,
        "feature": joint_row.feature,
        "feature_label": joint_row.feature_label,
        "family_match": joint_row.family_match,
        "task_negative_control": joint_row.task_negative_control,
        "joint_spearman_rho": joint_row.spearman_rho,
        "estimable_raters": len(rater_subset),
        "same_direction_raters": int(same_direction.sum()),
        "all_estimable_raters_same_direction": bool(
            len(rater_subset) > 0 and same_direction.all()
        ),
        "minimum_rater_rho": (
            float(rater_subset["spearman_rho"].min())
            if len(rater_subset)
            else np.nan
        ),
        "maximum_rater_rho": (
            float(rater_subset["spearman_rho"].max())
            if len(rater_subset)
            else np.nan
        ),
    })

rater_consistency = pd.DataFrame(rater_consistency_rows)
export_dataframe(
    rater_consistency,
    TABLE_DIR / "joint_vs_rater_descriptive_direction_consistency.csv",
)
display(rater_consistency.head(40))


## 11. Machine-indicator correlation and human-item co-occurrence structure

In [ ]:
feature_order = (
    feature_registry.assign(
        family_order=lambda frame: frame["family"].map(
            {family: index for index, family in enumerate(MACHINE_FAMILY_ORDER)}
        )
    )
    .sort_values(["family_order", "feature"])
    .reset_index(drop=True)
)
artifact_columns = [
    f"{feature}__artifact"
    for feature in feature_order["feature"]
]
machine_matrix = analysis_519.set_index("recording_id")[artifact_columns]
machine_spearman = machine_matrix.corr(method="spearman", min_periods=30)
machine_spearman.index = feature_order["label"].tolist()
machine_spearman.columns = feature_order["label"].tolist()
export_dataframe(
    machine_spearman.reset_index(names="feature_label"),
    TABLE_DIR / "machine_indicator_spearman_matrix.csv",
)

item_order = item_metadata["item"].tolist()
human_pivot = (
    human_item.pivot(
        index="recording_id",
        columns="item",
        values="present",
    )
    .reindex(columns=item_order)
    .astype(bool)
)
human_phi = human_pivot.astype(int).corr(method="pearson")

jaccard = pd.DataFrame(
    np.nan,
    index=item_order,
    columns=item_order,
)
for first in item_order:
    for second in item_order:
        union = (human_pivot[first] | human_pivot[second]).sum()
        intersection = (
            human_pivot[first] & human_pivot[second]
        ).sum()
        jaccard.loc[first, second] = (
            intersection / union if union > 0 else np.nan
        )

export_dataframe(
    human_phi.reset_index(names="item"),
    TABLE_DIR / "human_item_phi_matrix.csv",
)
export_dataframe(
    jaccard.reset_index(names="item"),
    TABLE_DIR / "human_item_jaccard_matrix.csv",
)


## 12. Mapping summaries and supplementary figures

In [ ]:
joint_raw = raw_correlation_atlas.loc[
    raw_correlation_atlas["scope"].eq("joint_unadjusted")
].copy()

within_family_raw = joint_raw.loc[
    joint_raw["family_match"]
    & ~joint_raw["task_negative_control"]
].copy()
off_family_raw = joint_raw.loc[
    ~joint_raw["family_match"]
    & ~joint_raw["task_negative_control"]
].copy()
task_raw = joint_raw.loc[
    joint_raw["task_negative_control"]
].copy()

export_dataframe(
    within_family_raw,
    TABLE_DIR / "raw_within_family_item_correlations.csv",
)
export_dataframe(
    off_family_raw,
    TABLE_DIR / "raw_off_family_item_correlations.csv",
)
export_dataframe(
    task_raw,
    TABLE_DIR / "raw_task_negative_control_correlations.csv",
)

mapping_summary = pd.DataFrame([
    {
        "relationship_class": "within-family acquisition items",
        "pairs": len(within_family_raw),
        "finite_correlations": int(within_family_raw["spearman_rho"].notna().sum()),
        "median_absolute_rho": float(within_family_raw["spearman_rho"].abs().median()),
        "adjusted_models_fit": int(
            adjusted_within_family["model_status"].eq("fit").sum()
        ),
        "adjusted_q_lt_0_05": int(
            adjusted_within_family["q_global_within_family_atlas"].lt(0.05).sum()
        ),
    },
    {
        "relationship_class": "off-family acquisition items",
        "pairs": len(off_family_raw),
        "finite_correlations": int(off_family_raw["spearman_rho"].notna().sum()),
        "median_absolute_rho": float(off_family_raw["spearman_rho"].abs().median()),
        "adjusted_models_fit": np.nan,
        "adjusted_q_lt_0_05": np.nan,
    },
    {
        "relationship_class": "task negative-control items",
        "pairs": len(task_raw),
        "finite_correlations": int(task_raw["spearman_rho"].notna().sum()),
        "median_absolute_rho": float(task_raw["spearman_rho"].abs().median()),
        "adjusted_models_fit": np.nan,
        "adjusted_q_lt_0_05": np.nan,
    },
])
export_dataframe(
    mapping_summary,
    TABLE_DIR / "mapping_class_summary.csv",
)
display(mapping_summary)

mpl.rcParams.update({
    "font.family": ["Arial", "DejaVu Sans"],
    "font.size": 7.0,
    "axes.titlesize": 8.5,
    "axes.labelsize": 7.3,
    "xtick.labelsize": 6.2,
    "ytick.labelsize": 6.2,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


def save_figure(figure: mpl.figure.Figure, stem: str) -> None:
    for suffix in ["png", "svg"]:
        figure.savefig(
            FIGURE_DIR / f"{stem}.{suffix}",
            dpi=400 if suffix == "png" else None,
            bbox_inches="tight",
            facecolor="white",
        )


def pivot_values(
    frame: pd.DataFrame,
    rows: list[str],
    columns: list[str],
    value: str,
) -> np.ndarray:
    pivot = frame.pivot_table(
        index="feature",
        columns="human_item",
        values=value,
        aggfunc="first",
    )
    return pivot.reindex(
        index=rows,
        columns=columns,
    ).to_numpy(dtype=float)

# Figure 1: all acquisition items × all indicators, raw descriptive rho.
acquisition_items = item_metadata.loc[
    item_metadata["signal_family"].ne("TASK"),
    "item",
].tolist()
feature_names = feature_order["feature"].tolist()
feature_labels = feature_order["label"].tolist()
raw_acquisition = joint_raw.loc[
    joint_raw["human_item"].isin(acquisition_items)
].copy()
rho_matrix = pivot_values(
    raw_acquisition,
    feature_names,
    acquisition_items,
    "spearman_rho",
)

figure, axis = plt.subplots(
    figsize=(14.5, 8.7),
    constrained_layout=True,
)
image = axis.imshow(
    np.ma.masked_invalid(rho_matrix),
    aspect="auto",
    cmap="RdBu_r",
    vmin=-0.35,
    vmax=0.35,
)
axis.set_xticks(np.arange(len(acquisition_items)))
axis.set_xticklabels(
    acquisition_items,
    rotation=55,
    ha="right",
)
axis.set_yticks(np.arange(len(feature_labels)))
axis.set_yticklabels([
    f"{family}  {label}"
    for family, label in zip(
        feature_order["family"],
        feature_labels,
    )
])
axis.set_title(
    "Descriptive human-item × signal-indicator correlation atlas",
    loc="left",
    fontweight="bold",
)
axis.set_xlabel("Human annotation item")
axis.set_ylabel("Artifact-oriented signal indicator")

for row_index, feature in enumerate(feature_names):
    feature_family = feature_order.loc[
        feature_order["feature"].eq(feature),
        "family",
    ].iloc[0]
    for column_index, item in enumerate(acquisition_items):
        item_family = item_metadata.loc[
            item_metadata["item"].eq(item),
            "signal_family",
        ].iloc[0]
        if feature_family == item_family:
            axis.add_patch(
                mpl.patches.Rectangle(
                    (column_index - 0.5, row_index - 0.5),
                    1,
                    1,
                    fill=False,
                    edgecolor="black",
                    linewidth=0.7,
                )
            )

colorbar = figure.colorbar(
    image,
    ax=axis,
    fraction=0.022,
    pad=0.015,
)
colorbar.set_label("Unadjusted Spearman ρ")
axis.text(
    0.0,
    -0.29,
    "Outlined cells are within-family. The full atlas is descriptive; adjusted inference is restricted to within-family cells.",
    transform=axis.transAxes,
)
save_figure(
    figure,
    "Supplementary_complete_raw_item_feature_correlation_atlas",
)
plt.show()
export_dataframe(
    raw_acquisition,
    FIGURE_SOURCE_DIR / "complete_raw_item_feature_correlation_atlas.csv",
)

# Figure 2: adjusted within-family atlas.
fit_adjusted = adjusted_within_family.loc[
    adjusted_within_family["model_status"].eq("fit")
].copy()
family_panels = [
    family
    for family in MACHINE_FAMILY_ORDER
    if (
        item_metadata["signal_family"].eq(family).any()
        and feature_registry["family"].eq(family).any()
    )
]
figure, axes = plt.subplots(
    2,
    3,
    figsize=(12.8, 7.5),
    constrained_layout=True,
)
axes = axes.ravel()
last_image = None
for axis, family in zip(axes, family_panels):
    family_items = item_metadata.loc[
        item_metadata["signal_family"].eq(family),
        "item",
    ].tolist()
    family_features = feature_order.loc[
        feature_order["family"].eq(family),
        "feature",
    ].tolist()
    family_feature_labels = feature_order.loc[
        feature_order["family"].eq(family),
        "label",
    ].tolist()
    family_table = fit_adjusted.loc[
        fit_adjusted["human_signal_family"].eq(family)
    ]
    matrix = pivot_values(
        family_table,
        family_features,
        family_items,
        "estimate",
    )
    q_matrix = pivot_values(
        family_table,
        family_features,
        family_items,
        "q_global_within_family_atlas",
    )
    last_image = axis.imshow(
        np.ma.masked_invalid(np.clip(matrix, -1.25, 1.25)),
        aspect="auto",
        cmap="RdBu_r",
        vmin=-1.25,
        vmax=1.25,
    )
    axis.set_xticks(np.arange(len(family_items)))
    axis.set_xticklabels(
        family_items,
        rotation=40,
        ha="right",
    )
    axis.set_yticks(np.arange(len(family_feature_labels)))
    axis.set_yticklabels(family_feature_labels)
    axis.set_title(family, loc="left", fontweight="bold")
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            if (
                np.isfinite(q_matrix[row_index, column_index])
                and q_matrix[row_index, column_index] < 0.05
            ):
                axis.text(
                    column_index,
                    row_index,
                    "•",
                    ha="center",
                    va="center",
                    fontsize=8,
                )

if last_image is not None:
    colorbar = figure.colorbar(
        last_image,
        ax=axes.tolist(),
        fraction=0.018,
        pad=0.015,
    )
    colorbar.set_label(
        "Adjusted artifact-oriented log-odds coefficient"
    )
figure.suptitle(
    "Adjusted within-family human-item mapping",
    fontsize=10,
    fontweight="bold",
)
save_figure(
    figure,
    "Supplementary_adjusted_within_family_item_mapping",
)
plt.show()
export_dataframe(
    adjusted_within_family,
    FIGURE_SOURCE_DIR / "adjusted_within_family_item_mapping.csv",
)

# Figure 3: joint and rater-specific raw correlations for strongest
# within-family mappings.
selected_pairs = fit_adjusted.loc[
    fit_adjusted["p_value"].lt(0.10)
    | fit_adjusted["wald_z"].abs().ge(1.96)
].sort_values("p_value").head(25)

columns = ["Joint", *EXPECTED_RATERS]
correlation_matrix = np.full(
    (len(selected_pairs), len(columns)),
    np.nan,
)
pair_labels = []
for row_index, row in enumerate(selected_pairs.itertuples(index=False)):
    pair_labels.append(
        f"{row.human_signal_family} | {row.human_item} ← {row.feature_label}"
    )
    for column_index, rater in enumerate(columns):
        match = raw_correlation_atlas.loc[
            raw_correlation_atlas["rater"].eq(rater)
            & raw_correlation_atlas["human_item"].eq(row.human_item)
            & raw_correlation_atlas["feature"].eq(row.feature)
        ]
        if len(match):
            correlation_matrix[row_index, column_index] = float(
                match.iloc[0]["spearman_rho"]
            )

figure, axis = plt.subplots(
    figsize=(8.8, max(4.5, 0.34 * len(selected_pairs))),
    constrained_layout=True,
)
image = axis.imshow(
    np.ma.masked_invalid(correlation_matrix),
    aspect="auto",
    cmap="RdBu_r",
    vmin=-0.45,
    vmax=0.45,
)
axis.set_xticks(np.arange(len(columns)))
axis.set_xticklabels(columns)
axis.set_yticks(np.arange(len(pair_labels)))
axis.set_yticklabels(pair_labels)
axis.set_title(
    "Joint and rater-stratified descriptive correlations",
    loc="left",
    fontweight="bold",
)
colorbar = figure.colorbar(
    image,
    ax=axis,
    fraction=0.03,
    pad=0.02,
)
colorbar.set_label("Unadjusted Spearman ρ")
axis.text(
    0.0,
    -0.08,
    "Rater columns describe separate recording subsets and are not direct tests of inter-rater differences.",
    transform=axis.transAxes,
)
save_figure(
    figure,
    "Supplementary_joint_and_rater_stratified_item_correlations",
)
plt.show()

# Figure 4: human item prevalence by rater.
prevalence_items = item_metadata["item"].tolist()
prevalence_matrix = (
    item_prevalence.set_index("item")
    .reindex(prevalence_items)[
        [f"prevalence_{rater}" for rater in EXPECTED_RATERS]
    ]
    .to_numpy(dtype=float)
)
figure, axis = plt.subplots(
    figsize=(6.8, 8.8),
    constrained_layout=True,
)
image = axis.imshow(
    prevalence_matrix,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="viridis",
)
axis.set_xticks(np.arange(len(EXPECTED_RATERS)))
axis.set_xticklabels(EXPECTED_RATERS)
axis.set_yticks(np.arange(len(prevalence_items)))
axis.set_yticklabels(prevalence_items)
axis.set_title(
    "Human item prevalence by production rater",
    loc="left",
    fontweight="bold",
)
colorbar = figure.colorbar(
    image,
    ax=axis,
    fraction=0.03,
    pad=0.02,
)
colorbar.set_label("Positive-label prevalence")
save_figure(
    figure,
    "Supplementary_human_item_prevalence_by_rater",
)
plt.show()

# Figure 5: machine correlation structure.
figure, axis = plt.subplots(
    figsize=(9.0, 8.2),
    constrained_layout=True,
)
image = axis.imshow(
    machine_spearman.to_numpy(),
    aspect="auto",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
)
axis.set_xticks(np.arange(len(feature_labels)))
axis.set_xticklabels(
    feature_labels,
    rotation=55,
    ha="right",
)
axis.set_yticks(np.arange(len(feature_labels)))
axis.set_yticklabels(feature_labels)
axis.set_title(
    "Spearman correlation among artifact-oriented signal indicators",
    loc="left",
    fontweight="bold",
)
colorbar = figure.colorbar(
    image,
    ax=axis,
    fraction=0.03,
    pad=0.02,
)
colorbar.set_label("Spearman ρ")
save_figure(
    figure,
    "Supplementary_machine_indicator_correlation",
)
plt.show()

# Figure 6: human acquisition-item co-occurrence.
acquisition_jaccard = jaccard.loc[
    acquisition_items,
    acquisition_items,
]
figure, axis = plt.subplots(
    figsize=(10.2, 9.2),
    constrained_layout=True,
)
image = axis.imshow(
    acquisition_jaccard.to_numpy(),
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="viridis",
)
axis.set_xticks(np.arange(len(acquisition_items)))
axis.set_xticklabels(
    acquisition_items,
    rotation=55,
    ha="right",
)
axis.set_yticks(np.arange(len(acquisition_items)))
axis.set_yticklabels(acquisition_items)
axis.set_title(
    "Co-occurrence of human acquisition-quality items",
    loc="left",
    fontweight="bold",
)
colorbar = figure.colorbar(
    image,
    ax=axis,
    fraction=0.03,
    pad=0.02,
)
colorbar.set_label("Jaccard similarity")
save_figure(
    figure,
    "Supplementary_human_item_cooccurrence",
)
plt.show()


## 13. Completion audit and manifest

In [ ]:
blocking_checks = pd.DataFrame([
    {
        "check": "Frozen recording population reproduced",
        "passed": len(analysis_519) == EXPECTED_FROZEN_RECORDINGS,
        "detail": f"n={len(analysis_519)}",
    },
    {
        "check": "Frozen participant count reproduced",
        "passed": (
            analysis_519["participant_id"].nunique()
            == EXPECTED_FROZEN_PARTICIPANTS
        ),
        "detail": f"n={analysis_519['participant_id'].nunique()}",
    },
    {
        "check": "Production annotation population reproduced",
        "passed": len(production_ids) == EXPECTED_HUMAN_RECORDINGS,
        "detail": f"n={len(production_ids)}",
    },
    {
        "check": "All 28 human items represented",
        "passed": len(item_metadata) == EXPECTED_HUMAN_ITEMS,
        "detail": f"n={len(item_metadata)}",
    },
    {
        "check": "All 19 machine indicators represented",
        "passed": len(feature_registry) == EXPECTED_MACHINE_FEATURES,
        "detail": f"n={len(feature_registry)}",
    },
    {
        "check": "Complete raw joint and rater correlation atlas exported",
        "passed": len(raw_correlation_atlas) == expected_raw_rows,
        "detail": f"rows={len(raw_correlation_atlas)}",
    },
    {
        "check": "Adjusted inference restricted to within-family cells",
        "passed": bool(
            (
                adjusted_within_family["human_signal_family"]
                == adjusted_within_family["machine_family"]
            ).all()
        ),
        "detail": f"pairs={len(adjusted_within_family)}",
    },
    {
        "check": "Finalized Stage-2 models independently reproduced",
        "passed": regression_verification["passed"].all(),
        "detail": f"pairs={len(regression_verification)}",
    },
    {
        "check": "Unsupported adjusted models not interpreted",
        "passed": adjusted_within_family.loc[
            ~adjusted_within_family["support_pass"],
            "model_status",
        ].eq("support_gate_failed").all(),
        "detail": "support gates enforced",
    },
    {
        "check": "Rater-specific results remain descriptive diagnostics",
        "passed": True,
        "detail": f"correlation_rows={len(raw_correlation_atlas.loc[raw_correlation_atlas['scope'].eq('rater_unadjusted')])}",
    },
    {
        "check": "No imputation performed",
        "passed": True,
        "detail": "pairwise complete-case availability retained",
    },
])

export_dataframe(
    blocking_checks,
    LOG_DIR / "mapping_atlas_blocking_completion_checklist.csv",
)
display(blocking_checks)

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "analysis_name": "Human-machine item mapping atlas",
    "frozen_recordings": len(analysis_519),
    "frozen_participants": int(
        analysis_519["participant_id"].nunique()
    ),
    "production_human_recordings": len(production_ids),
    "human_items": len(item_metadata),
    "machine_indicators": len(feature_registry),
    "raw_correlation_rows": len(raw_correlation_atlas),
    "adjusted_within_family_pairs": len(adjusted_within_family),
    "adjusted_within_family_models_fit": int(
        adjusted_within_family["model_status"].eq("fit").sum()
    ),
    "rater_descriptive_correlation_rows": int(raw_correlation_atlas["scope"].eq("rater_unadjusted").sum()),
    "stage2_regression_pairs_verified": len(regression_verification),
    "final_run": FINAL_RUN,
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "statsmodels": sm.__version__,
}
(
    MANIFEST_DIR / "mapping_atlas_run_manifest.json"
).write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

if not blocking_checks["passed"].all():
    print(
        "\nMAPPING ATLAS COMPLETED WITH BLOCKING ITEMS. "
        "Do not interpret outputs."
    )
else:
    print(
        "\nMAPPING ATLAS COMPLETED. "
        "All exploratory outputs are ready for scientific review."
    )


---

## Interpretation boundary

The complete correlation atlas is descriptive. It shows how individual human items and artifact-oriented signal indicators co-vary, jointly and within each rater's separate recording subset. Adjusted inference is restricted to within-family relationships and uses logistic models with participant-clustered sandwich covariance and rater, cohort, and recording-duration adjustment. Rater-stratified correlations are descriptive rather than direct comparisons of rater performance because production recordings were not rated by all four raters. Unsupported cells remain blank rather than being interpreted as null effects.
